# RAG Resolver - semantic memory primitives

`%run resolver.ipynb` (the lexical base) and adds the RAG primitives ONLY: build text
chunks over the memory store, embed them behind a pluggable `EmbeddingProvider`, cache a
per-user index, and `semantic_search`. The lexical + RAG *integration* (hybrid resolve,
evals, threshold tuning) lives in `hybrid_resolver.ipynb`; orchestration in `pipeline.ipynb`.

In [1]:
import os
# Run from backend/ so `%run` resolves regardless of the kernel's launch dir.
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
_RAG_DEMO_TOP = globals().get("RUN_DEMO", True)
_RAG_LIVE_TOP = globals().get("RUN_LIVE", False)
RUN_DEMO = RUN_LIVE = False   # suppress the lexical resolver's own demo + live cell while loading it
%run resolver.ipynb
RUN_DEMO, RUN_LIVE = _RAG_DEMO_TOP, _RAG_LIVE_TOP

common ready. cwd=backend TODAY=2026-06-16
TalkingPoint, TopicTheme, PlannerOutput, BrainstormOutput, render_brainstorm defined.
Person, QueryClues, Candidate, ResolutionResult, Interaction, Fact, PrepSession defined.
store ready: u1 has 6 people, 1 interactions.
name matching (damerau_levenshtein, soundex, name_match_score) defined.
generate_candidates defined.
decide + resolve_from_clues defined.
clues_agent + extract_clues defined.
resolve_meeting_context + show_resolution defined.
RouteDecision + route_resolution defined.
select_candidate defined.
write-back defined: save_prep_session / get_prep_sessions / propose_new_person / save_new_person / find_merge_candidates / apply_merge / propose_fact / add_fact
intent_agent + classify_intent defined.
recall (render_recall_answer/apply_recall/run_recall) + update (extract_update/apply_update/run_update) defined.


In [2]:
# RAG Step 1 - memory chunk schemas + chunk builder (text views over the store).
# A chunk is self-contained memory text that can be embedded and retrieved;
# person_id is preserved so a hit maps back to a contact.
class MemoryChunk(BaseModel):
    id: str
    user_id: str
    target_type: Literal["person", "interaction", "fact", "prep_session"]
    target_id: str
    person_id: str | None = None
    text: str
    metadata: dict = Field(default_factory=dict)
    embedding: list[float] | None = None
    updated_at: str = ""


class RagHit(BaseModel):
    chunk_id: str
    target_id: str
    person_id: str | None = None
    score: float
    text: str
    metadata: dict = Field(default_factory=dict)


def build_memory_chunks(user_id: str) -> list[MemoryChunk]:
    """One chunk per person profile, confirmed fact, interaction, and prep session."""
    now = datetime.datetime.now().isoformat(timespec="seconds")
    people = get_people(user_id)
    name_of = {p.id: p.name for p in people}
    chunks: list[MemoryChunk] = []

    for p in people:
        text = p.name + (f", {p.role}" if p.role else "")
        if p.tags:
            text += ". Interests: " + ", ".join(p.tags)
        if p.profile_summary:
            text += ". " + p.profile_summary
        chunks.append(MemoryChunk(
            id=f"person:{p.id}", user_id=user_id, target_type="person",
            target_id=p.id, person_id=p.id, text=text, updated_at=now,
            metadata={"person_name": p.name, "source": "person", "tags": p.tags}))
        for i, f in enumerate(p.facts):
            chunks.append(MemoryChunk(
                id=f"fact:{p.id}:{i}", user_id=user_id, target_type="fact",
                target_id=p.id, person_id=p.id, text=f"{p.name} {f.text}", updated_at=f.observed_at,
                metadata={"person_name": p.name, "source": "fact", "observed_at": f.observed_at}))

    for it in get_interactions(user_id):
        text = (it.summary or it.raw_note)
        if it.topics:
            text += ". Topics: " + ", ".join(it.topics)
        chunks.append(MemoryChunk(
            id=f"interaction:{it.id}", user_id=user_id, target_type="interaction",
            target_id=it.id, person_id=it.person_id, text=text, updated_at=it.date or now,
            metadata={"person_name": name_of.get(it.person_id), "source": "interaction",
                      "date": it.date, "topics": it.topics}))

    for ps in get_prep_sessions(user_id):
        pid = ps.target_ids[0] if ps.target_ids else None
        chunks.append(MemoryChunk(
            id=f"prep_session:{ps.id}", user_id=user_id, target_type="prep_session",
            target_id=ps.id, person_id=pid, text=f"Prep session: {ps.user_query}",
            updated_at=ps.timestamp, metadata={"person_name": name_of.get(pid), "source": "prep_session"}))

    return chunks


print("MemoryChunk + RagHit + build_memory_chunks defined.")

MemoryChunk + RagHit + build_memory_chunks defined.


In [3]:
# RAG Step 2 - embeddings behind a pluggable provider + local cosine index (cached).
# OpenAI is the default provider (it has a real embeddings endpoint; OpenRouter does
# not). A local, no-API provider can be dropped in without touching the resolver.
import numpy as np
from typing import Protocol

EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")


class EmbeddingProvider(Protocol):
    name: str
    async def embed(self, texts: list[str]) -> list[list[float]]: ...


class OpenAIEmbeddings:
    name = "openai"

    def __init__(self, api_key: str, model: str = EMBED_MODEL):
        self.model = model
        self._client = AsyncOpenAI(api_key=api_key)

    async def embed(self, texts: list[str]) -> list[list[float]]:
        r = await self._client.embeddings.create(model=self.model, input=texts)
        return [d.embedding for d in r.data]


# Pluggable slot for a local, no-API provider (drop in without touching the resolver):
#   class LocalSTEmbeddings:
#       name = "local-st"
#       def __init__(self, model="all-MiniLM-L6-v2"):
#           from sentence_transformers import SentenceTransformer   # uv add sentence-transformers
#           self._m = SentenceTransformer(model)
#       async def embed(self, texts):
#           return [v.tolist() for v in self._m.encode(texts, normalize_embeddings=True)]


def get_embedding_provider(api_key: str | None = None) -> EmbeddingProvider | None:
    """Select a provider from config. No key -> None (RAG disabled, lexical fallback)."""
    key = api_key if api_key is not None else os.environ.get("OPENAI_API_KEY")
    return OpenAIEmbeddings(api_key=key) if key else None


embedding_provider = get_embedding_provider()
RAG_ENABLED = embedding_provider is not None
if not RAG_ENABLED:
    print("[rag] no embedding provider -> RAG disabled; the lexical resolver still works.")


async def embed_text(text: str) -> list[float]:
    return (await embedding_provider.embed([text]))[0]


async def embed_chunks(chunks: list[MemoryChunk]) -> list[MemoryChunk]:
    """Embed each chunk's text in place (one batched call). Empty list is a no-op."""
    if not chunks:
        return chunks
    vecs = await embedding_provider.embed([c.text for c in chunks])
    for c, v in zip(chunks, vecs):
        c.embedding = v
    return chunks


def cosine_similarity(a: list[float], b: list[float]) -> float:
    va, vb = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    na, nb = np.linalg.norm(va), np.linalg.norm(vb)
    return 0.0 if na == 0 or nb == 0 else float(va @ vb / (na * nb))


def rank_chunks(query_vec: list[float], chunks: list[MemoryChunk], k: int = 8) -> list[RagHit]:
    """Pure ranking (no LLM). Chunks without an embedding are skipped, not errored."""
    scored = [(cosine_similarity(query_vec, c.embedding), c) for c in chunks if c.embedding]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [RagHit(chunk_id=c.id, target_id=c.target_id, person_id=c.person_id,
                   score=round(s, 4), text=c.text, metadata=c.metadata)
            for s, c in scored[:k]]


# Cached per-user embedded index: embed once, rebuild after a memory write.
_RAG_INDEX: dict[str, list[MemoryChunk]] = {}


async def index_memory(user_id: str, rebuild: bool = False) -> list[MemoryChunk]:
    if rebuild or user_id not in _RAG_INDEX:
        _RAG_INDEX[user_id] = await embed_chunks(build_memory_chunks(user_id))
    return _RAG_INDEX[user_id]


async def semantic_search(user_id: str, query: str, k: int = 8) -> list[RagHit]:
    """Embed the query, rank the user's cached memory index by cosine similarity.
    Returns [] when RAG is disabled so callers can fall back to lexical."""
    if not RAG_ENABLED:
        return []
    chunks = await index_memory(user_id)
    return rank_chunks(await embed_text(query), chunks, k)


print(f"RAG layer ready (provider={embedding_provider.name if embedding_provider else None}): "
      "embed_text / embed_chunks / cosine_similarity / rank_chunks / index_memory / semantic_search.")

RAG layer ready (provider=openai): embed_text / embed_chunks / cosine_similarity / rank_chunks / index_memory / semantic_search.


In [4]:
# Demo - offline checks for the chunk builder, cosine ranking, and provider selection.
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    fails = 0

    def _check(label, got, want):
        global fails
        ok = got == want
        fails += 0 if ok else 1
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}: {got!r} (want {want!r})")

    print("RAG Step 1 - memory chunks:")
    PEOPLE["u_rag"] = [Person(id="p_a", name="Ann", role="designer", tags=["chess"], profile_summary="A friend.")]
    add_fact("u_rag", "p_a", propose_fact("is into pottery"))
    INTERACTIONS["u_rag"] = [Interaction(id="i_a", person_id="p_a", summary="Talked about hiking", topics=["hiking"])]
    ch = build_memory_chunks("u_rag")
    _check("RS1a person/fact/interaction chunks", {"person", "fact", "interaction"} <= {c.target_type for c in ch}, True)
    _check("RS1b person_id preserved", all(c.person_id == "p_a" for c in ch), True)
    _check("RS1c chunk text non-empty", all(c.text.strip() for c in ch), True)
    _check("RS1d empty user -> no chunks", build_memory_chunks("nobody_xyz"), [])

    print("RAG Step 2 - cosine + ranking + provider (offline):")
    _check("RS2a cosine identical = 1", round(cosine_similarity([1.0, 0.0, 0.0], [1.0, 0.0, 0.0]), 3), 1.0)
    _check("RS2b cosine ranks similar higher",
           cosine_similarity([1.0, 0.0, 0.0], [0.9, 0.1, 0.0]) > cosine_similarity([1.0, 0.0, 0.0], [0.0, 0.0, 1.0]), True)
    _ck = [MemoryChunk(id="x", user_id="t", target_type="person", target_id="p1", text="a", embedding=[1.0, 0.0]),
           MemoryChunk(id="y", user_id="t", target_type="person", target_id="p2", text="b", embedding=None)]
    _hits = rank_chunks([1.0, 0.0], _ck, k=5)
    _check("RS2c missing-embedding chunk skipped", [h.chunk_id for h in _hits], ["x"])
    _check("RS2d hit carries target_id", _hits[0].target_id, "p1")
    _check("RS2e RAG_ENABLED reflects provider", RAG_ENABLED, embedding_provider is not None)
    _check("RS2f provider name", embedding_provider.name if embedding_provider else None, "openai")
    _check("RS2g no-key -> no provider (lexical fallback)", get_embedding_provider(api_key="") is None, True)

    print(f"\n{'ALL PASS' if fails == 0 else str(fails) + ' FAILURE(S)'}")

RAG Step 1 - memory chunks:
  [PASS] RS1a person/fact/interaction chunks: True (want True)
  [PASS] RS1b person_id preserved: True (want True)
  [PASS] RS1c chunk text non-empty: True (want True)
  [PASS] RS1d empty user -> no chunks: [] (want [])
RAG Step 2 - cosine + ranking + provider (offline):
  [PASS] RS2a cosine identical = 1: 1.0 (want 1.0)
  [PASS] RS2b cosine ranks similar higher: True (want True)
  [PASS] RS2c missing-embedding chunk skipped: ['x'] (want ['x'])
  [PASS] RS2d hit carries target_id: 'p1' (want 'p1')
  [PASS] RS2e RAG_ENABLED reflects provider: True (want True)
  [PASS] RS2f provider name: 'openai' (want 'openai')
  [PASS] RS2g no-key -> no provider (lexical fallback): True (want True)

ALL PASS


In [5]:
# Live demo - semantic_search over the memory index (real embeddings). OFF by default.
RUN_LIVE = globals().get("RUN_LIVE", False)
if RUN_LIVE and RAG_ENABLED:
    print("LIVE RAG - semantic_search over u1:")
    for q in ["person who talked about ML grad programs", "the friend who bakes bread"]:
        hits = await semantic_search("u1", q, k=2)
        print(f"  Q: {q}")
        for h in hits:
            print(f"     {h.score:.3f}  {h.metadata.get('person_name')!s:8} [{h.metadata.get('source')}]")
elif RUN_LIVE:
    print("RUN_LIVE set but RAG disabled (no OPENAI_API_KEY).")